### Working with PlaywrightTookit

In [ ]:
%pip install --upgrade --quiet  playwright
%pip install -qU  playwright > /dev/null
%pip install -qU  lxml
!pip install playwright
!playwright install

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3-coder:480b-cloud", # Ollama supporting some cloud models which takes less memory and responsd faster
    temperature=0.5
)

In [ ]:
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import (
    create_async_playwright_browser,
)
import nest_asyncio
nest_asyncio.apply()

- nest_asyncio is a Python library that lets you run async code inside environments that already have an event loop (like Jupyter notebooks).
- Without this, you might get errors like “event loop already running” when trying to run async Playwright code.
- nest_asyncio.apply() patches the event loop so you can safely run async Playwright commands in interactive environments.


##### Get all the Tools from PlaywrightToolkit

In [ ]:
async_browser = create_async_playwright_browser()
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()
tools

#This code creates a Playwright browser, wraps it in a LangChain toolkit, and extracts ready-to-use browser automation tools for an agent.

: 

In [ ]:
tool_by_name = {tool.name: tool for tool in tools}
navigate_tool = tool_by_name["navigate_browser"]
get_elements = tool_by_name["get_elements"]
navigate_tool

#This code builds a dictionary of browser tools and retrieves the two tools needed for navigation and extracting elements from a webpage.

In [ ]:
await navigate_tool.arun({"url": "http://eaapp.somee.com/Employee"})

#await tells Python: “This function is asynchronous. Pause here, let other tasks run, and come back when the result is ready.”

- run()
    - Used for normal (sync) code
    - Blocks until the tool finishes

- arun()
    - Used for async code
    - Must be called with await
    - Doesn’t block the whole program
    - Works inside async functions or async Jupyter cells

In [ ]:
await get_elements.arun({"selector": "td", "attributes":["innerText"]})
# This command extracts the text from all table cells (<td>) on the webpage using Playwright’s async get-elements tool.

### Working with AI Agents

In [ ]:
from langchain_classic.agents import initialize_agent, AgentType

# Creating an AI Agent
agent = initialize_agent(
    tools= tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

query = "Extract the table data from http://eaapp.somee.com/Employee page and get me the average salary of the employees"

result = await agent.arun(query)

print(result)

#This code creates an AI agent with browser automation tools and asks it to visit a webpage, 
# extract table data, and compute the average employee salary — all using asynchronous execution.
# verbose=True makes the agent show all its internal actions and tool interactions so you can see exactly what it’s doing.

In [ ]:
print(result)

### Agent with Langchain and PlaywrightToolkit v1.0.5

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Creating an AI Agent
agent = create_agent(
    tools= tools,
    model=llm
)

query = "Extract the table data from http://eaapp.somee.com/Employee page and get me the average salary of the employees"

result = await agent.ainvoke({"messages": [HumanMessage(content=query)]})

print(result["messages"][-1].content)

#This code creates a modern LangChain agent with browser tools, asks it to extract table data from a webpage and compute the average salary, 
# runs it asynchronously, and prints the final answer.

- create_agent (new API)
- initialize_agent (old API):